# Phase 1: Image Ingestion and Document Flattening
### Comprehensive Technical Explanation & Interactive Walkthrough
**System:** Camera-Based Handwriting Recognition  
**Architecture:** Classical Computer Vision + Shallow Multi-Layer Perceptron (MLP)  
**Document Reference:** `Informations/Camera_Based_Handwriting_Recognition_Antigravity_Spec.pdf`  

---

## 1. Motivation and Objectives of Phase 1

When performing optical character recognition on handwritten documents captured via smartphone cameras or webcams, the input image almost never resembles a flat, clean scanner output. Instead, camera captures suffer from:
1. **Perspective Distortion (Keystone Effect):** The camera sensor is rarely perfectly parallel to the paper surface, causing text lines to converge and characters to shrink with distance.
2. **Rotational Misalignment:** The paper may be placed at an arbitrary orientation on a desk.
3. **Cluttered Environmental Surroundings:** Desk textures, pens, fingers, shadows, and table edges surround the document.
4. **Variable Resolutions & File Corruption:** Input files may have varying dimensions, color spaces, or missing/corrupted file headers.

### Phase 1 Mandates from Project Specification:
* **Safe Ingestion:** Read images using `cv2.imread` with comprehensive file existence and format decoding checks, reporting unreadable files clearly.
* **Untouched Preservation:** Retain an untouched, deep-copied duplicate of the original photograph so that later visual verification overlays can be rendered without mutating raw input pixels.
* **Optional 4-Point Perspective Transform:** Detect paper boundaries and warp the skewed document into a rectified, flat, top-down rectangular view.
* **Graceful Fallback:** If page boundaries cannot be identified (e.g. a tight close-up crop of notebook lines), automatically bypass the transform without crashing or corrupting image data.

## 2. Environment Setup & Library Imports

In strict compliance with the **Zero Deep Learning Policy**, Phase 1 relies exclusively on:
* `cv2` (OpenCV): High-performance classical computer vision primitives for image loading, color spaces, edge filtering, contour geometry, and homography warps.
* `numpy` (NumPy): Fast array manipulation, linear algebra operations, Euclidean distance metrics, and coordinate sorting.
* `matplotlib.pyplot`: Visual display of intermediate edge maps and side-by-side diagnostic plots.
* `pathlib.Path` & `sys`: Standard cross-platform file validation and operating system abstractions.

In [ ]:
import os
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Verify runtime environment
print(f"Python Executable : {sys.executable}")
print(f"Python Version    : {sys.version.split()[0]}")
print(f"OpenCV Version    : {cv2.__version__}")
print(f"NumPy Version     : {np.__version__}")

## 3. Step 1: Safe Image Ingestion & Untouched Memory Preservation

### Algorithmic Detail:
1. **File Validation:** When reading from disk via `cv2.imread(filepath, cv2.IMREAD_COLOR)`, OpenCV does **not** raise an exception if a file does not exist or is corrupted; it silently returns `None`. Without proactive validation, this results in downstream `AttributeError: 'NoneType' object has no attribute 'shape'` errors.
2. **Error Reporting:** We verify that the path exists using `Path.exists()`, load the image, and explicitly verify `image is not None and image.size > 0`.
3. **Untouched Preservation:** In Python, assignments like `original = image` create shallow pointer references, meaning subsequent drawing operations (`cv2.rectangle`, `cv2.line`) would irreversibly contaminate the original photo. We use `image.copy()` to ensure a completely independent memory buffer.

In [ ]:
def safe_load_image(file_path):
    """
    Safely loads an image file from disk.
    Returns: (untouched_original, working_image)
    Raises descriptive exceptions on missing or corrupted files.
    """
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"[Phase 1] Target image not found at: '{file_path}'")
    
    # Read image in standard 3-channel BGR format
    image = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image is None or image.size == 0:
        raise ValueError(f"[Phase 1] Failed to decode image file. File may be damaged or in an unsupported format: '{file_path}'")
    
    # Guarantee deep memory copy for pristine preservation
    untouched_original = image.copy()
    return untouched_original, image

print("safe_load_image function defined successfully.")

## 4. Step 2: Document Boundary & Corner Detection

### Mathematical & Algorithmic Process:
To rectify a document placed on a surface, we must locate the 4 physical corners of the paper sheet:

$$\mathbf{P} = \{ (x_1, y_1), (x_2, y_2), (x_3, y_3), (x_4, y_4) \}$$

1. **Adaptive Downscaling:** Modern smartphones take 12MP to 48MP photos ($4000 \times 3000\text{px}$). Processing raw contours at this scale is computationally wasteful and introduces micro-noise. We scale the working copy so its longest dimension is at most $1000\text{px}$, noting the scaling factor $s = \frac{1000}{\max(H, W)}$.
2. **Edge-Focused Filtering:**
   * Convert to grayscale: $I_{\text{gray}} = 0.299 R + 0.587 G + 0.114 B$
   * Gaussian smoothing ($5 \times 5$ kernel, $\sigma=0$) to eliminate paper texture and dust.
   * Canny Edge Detection (thresholds $50, 150$) to locate high-gradient boundaries.
   * Morphological Dilation ($3 \times 3$ rectangular kernel) to bridge any micro-breaks in the paper outline.
3. **Contour Extraction & Polygon Approximation:**
   * Compute external contours using `cv2.findContours(..., cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)`.
   * Sort candidate contours by enclosed area in descending order.
   * Apply the **Douglas-Peucker algorithm** (`cv2.approxPolyDP`):
     $$\epsilon = 0.02 \times \text{Perimeter}(C)$$
   * Search for a convex contour (`cv2.isContourConvex`) with exactly **4 vertices**.
4. **Area Validation:**
   * Verify that the detected quadrilateral occupies at least $15\%$ of the total camera frame area (preventing false detection of small business cards or sticky notes).
   * Rescale the 4 corner points back to the original full-resolution space: $\mathbf{P}_{\text{orig}} = \frac{\mathbf{P}_{\text{small}}}{s}$.

In [ ]:
def detect_document_corners(image, min_area_ratio=0.15):
    """
    Locates the 4 corners of a rectangular document sheet in the input frame.
    Returns: (corners_array, success_boolean, debug_dict)
    """
    h, w = image.shape[:2]
    
    # 1. Downscale for speed and edge clarity
    max_dim = 1000
    scale = 1.0
    if max(h, w) > max_dim:
        scale = max_dim / float(max(h, w))
        small = cv2.resize(image, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    else:
        small = image.copy()
        
    # 2. Edge-focused filtering
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edged = cv2.Canny(blurred, 50, 150)
    dilated = cv2.dilate(edged, cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3)), iterations=1)
    
    # 3. Contour analysis
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    debug_data = {"small_image": small, "gray": gray, "edged": edged, "dilated": dilated}
    
    if not contours:
        return None, False, debug_data
        
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    small_area = small.shape[0] * small.shape[1]
    
    for c in contours[:5]:
        perimeter = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * perimeter, True)
        
        if len(approx) == 4 and cv2.isContourConvex(approx):
            area = cv2.contourArea(approx)
            if (area / small_area) >= min_area_ratio:
                # Rescale back to original resolution coordinates
                corners = (approx.reshape(4, 2) / scale).astype("float32")
                return corners, True, debug_data
                
    return None, False, debug_data

print("detect_document_corners defined successfully.")

## 5. Step 3: Deterministic Corner Ordering (TL, TR, BR, BL)

### Mathematical Formulation:
OpenCV contour detection returns quadrilateral vertices in arbitrary sequential order depending on where the contour tracing started. A homography mapping requires an exact, consistent point-to-point correspondence between source corners and destination rectangle corners:

$$\mathbf{P}_{\text{ordered}} = \begin{bmatrix} (x_{\text{TL}}, y_{\text{TL}}) \\ (x_{\text{TR}}, y_{\text{TR}}) \\ (x_{\text{BR}}, y_{\text{BR}}) \\ (x_{\text{BL}}, y_{\text{BL}}) \end{bmatrix}$$

We determine the role of each corner using linear combinations of $(x, y)$:

1. **Coordinate Sum ($S_i = x_i + y_i$):**
   * **Top-Left (TL):** Closest to origin $(0, 0) \implies$ **Minimizes** $(x + y)$.
   * **Bottom-Right (BR):** Furthest from origin $\implies$ **Maximizes** $(x + y)$.

2. **Coordinate Difference ($D_i = x_i - y_i$):**
   * **Top-Right (TR):** Large $x$, small $y \implies$ **Maximizes** $(x - y)$ (or minimizes $y - x$).
   * **Bottom-Left (BL):** Small $x$, large $y \implies$ **Minimizes** $(x - y)$ (or maximizes $y - x$).

In [ ]:
def order_points(pts):
    """
    Orders 4 quadrilateral coordinates deterministically:
    [Top-Left, Top-Right, Bottom-Right, Bottom-Left]
    """
    pts = np.array(pts, dtype="float32").reshape(4, 2)
    rect = np.zeros((4, 2), dtype="float32")
    
    # Sum: x + y
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]  # Top-Left (min sum)
    rect[2] = pts[np.argmax(s)]  # Bottom-Right (max sum)
    
    # Difference: x - y
    diff = np.diff(pts, axis=1)  # (y - x)
    rect[1] = pts[np.argmin(diff)]  # Top-Right (min y - x => max x - y)
    rect[3] = pts[np.argmax(diff)]  # Bottom-Left (max y - x => min x - y)
    
    return rect

# Example verification
test_pts = np.array([[300, 400], [100, 100], [400, 150], [80, 380]])
ordered = order_points(test_pts)
print("Arbitrary input points:\n", test_pts)
print("Deterministic order [TL, TR, BR, BL]:\n", ordered)

## 6. Step 4: Perspective Rectification & Homography Computation

### Homography Transform Mathematics:
A planar perspective transform represents a projective mapping between two planes via a $3 \times 3$ matrix $\mathbf{M}$:

$$\begin{bmatrix} x' \\ y' \\ 1 \end{bmatrix} \sim \mathbf{M} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} = \begin{bmatrix} m_{11} & m_{12} & m_{13} \\ m_{21} & m_{22} & m_{23} \\ m_{31} & m_{32} & m_{33} \end{bmatrix} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}$$

1. **Calculating Target Dimensions:**
   To prevent stretching or squashing the document, we calculate the rectified width $W_{\text{target}}$ and height $H_{\text{target}}$ using the maximum Euclidean distance of opposing edges:
   $$W_{\text{bottom}} = \|\mathbf{p}_{\text{BR}} - \mathbf{p}_{\text{BL}}\|_2, \quad W_{\text{top}} = \|\mathbf{p}_{\text{TR}} - \mathbf{p}_{\text{TL}}\|_2$$
   $$W_{\text{target}} = \max(\text{round}(W_{\text{bottom}}), \text{round}(W_{\text{top}}))$$
   $$H_{\text{right}} = \|\mathbf{p}_{\text{TR}} - \mathbf{p}_{\text{BR}}\|_2, \quad H_{\text{left}} = \|\mathbf{p}_{\text{TL}} - \mathbf{p}_{\text{BL}}\|_2$$
   $$H_{\text{target}} = \max(\text{round}(H_{\text{right}}), \text{round}(H_{\text{left}}))$$

2. **Destination Coordinate Grid:**
   $$\mathbf{P}_{\text{dst}} = \begin{bmatrix} (0, 0) \\ (W_{\text{target}}-1, 0) \\ (W_{\text{target}}-1, H_{\text{target}}-1) \\ (0, H_{\text{target}}-1) \end{bmatrix}$$

3. **Homography Solution & Warping:**
   * `M = cv2.getPerspectiveTransform(P_ordered, P_dst)` computes the 8 degrees of freedom in $\mathbf{M}$.
   * `cv2.warpPerspective(image, M, (W_target, H_target), flags=cv2.INTER_CUBIC)` produces the flat top-down surface with bicubic anti-aliasing.

In [ ]:
def four_point_transform(image, corners):
    """
    Warps a quadrilateral document region into a rectified flat rectangle.
    Returns: (warped_image, homography_matrix, (target_w, target_h))
    """
    rect = order_points(corners)
    (tl, tr, br, bl) = rect
    
    # Euclidean width
    width_bottom = np.linalg.norm(br - bl)
    width_top = np.linalg.norm(tr - tl)
    target_w = max(int(round(width_bottom)), int(round(width_top)))
    
    # Euclidean height
    height_right = np.linalg.norm(tr - br)
    height_left = np.linalg.norm(tl - bl)
    target_h = max(int(round(height_right)), int(round(height_left)))
    
    # Safety clamp against degenerate dimensions
    target_w = max(target_w, 100)
    target_h = max(target_h, 100)
    
    # Destination rectangular plane
    dst_pts = np.array([
        [0, 0],
        [target_w - 1, 0],
        [target_w - 1, target_h - 1],
        [0, target_h - 1]
    ], dtype="float32")
    
    # Compute 3x3 homography matrix
    M = cv2.getPerspectiveTransform(rect, dst_pts)
    
    # Apply warp
    warped = cv2.warpPerspective(image, M, (target_w, target_h), flags=cv2.INTER_CUBIC)
    return warped, M, (target_w, target_h)

print("four_point_transform defined successfully.")

## 7. Step 5: Graceful Bypass & Manual Fallback Architecture

A critical architectural requirement from the project brief is:
> *"Make the four-point perspective transform optional. If page corners cannot be detected reliably, allow the user to skip it or supply corners manually."*

### How Our Architecture Handles This:
1. **Manual Corners Override:** If the user supplies an explicit list of coordinates `manual_corners=[...]`, the engine directly executes the homography warp using those points without running heuristic contour detection.
2. **Automatic Detection Bypass:** If `enable_warp=True` but no clean quadrilateral is found (e.g. the camera is cropped closely inside the document where paper edges are off-screen), the engine:
   * Logs an informative notice: `"[Phase 1] Notice: No distinct page boundary detected. Gracefully bypassing warp."`
   * Sets `meta['warp_applied'] = False`
   * Passes the entire full frame directly into Phase 2 without distortion or crashing.
3. **Configurable Toggle:** If `enable_warp=False`, perspective detection is completely bypassed.

In [ ]:
class DocumentIngestionPipeline:
    """
    Unified Phase 1 Pipeline orchestrator.
    """
    def __init__(self, enable_warp=True, min_contour_area_ratio=0.15):
        self.enable_warp = enable_warp
        self.min_contour_area_ratio = min_contour_area_ratio

    def process(self, image_input, manual_corners=None):
        """
        Executes Phase 1 ingestion, returns (untouched_original, working_image, metadata)
        """
        if isinstance(image_input, (str, Path)):
            original_copy, working_image = safe_load_image(image_input)
        elif isinstance(image_input, np.ndarray):
            original_copy = image_input.copy()
            working_image = image_input.copy()
        else:
            raise TypeError(f"Unsupported image input type: {type(image_input)}")

        meta = {
            "warp_applied": False,
            "corners": None,
            "homography_matrix": None,
            "original_shape": original_copy.shape,
            "ingested_shape": None,
        }

        # 1. Manual corners check
        if manual_corners is not None:
            print("[Phase 1] Applying user-supplied manual corner coordinates.")
            warped, M, (tw, th) = four_point_transform(working_image, manual_corners)
            meta.update({"warp_applied": True, "corners": manual_corners, "homography_matrix": M, "ingested_shape": warped.shape})
            return original_copy, warped, meta

        # 2. Automatic warp if enabled
        if self.enable_warp:
            corners, success, _ = detect_document_corners(working_image, self.min_contour_area_ratio)
            if success:
                print("[Phase 1] Document boundary identified! Executing perspective rectification.")
                warped, M, (tw, th) = four_point_transform(working_image, corners)
                meta.update({"warp_applied": True, "corners": corners, "homography_matrix": M, "ingested_shape": warped.shape})
                return original_copy, warped, meta
            else:
                print("[Phase 1] Notice: No distinct quadrilateral found. Gracefully bypassing warp.")
        else:
            print("[Phase 1] Perspective warp disabled via configuration.")

        meta["ingested_shape"] = working_image.shape
        return original_copy, working_image, meta

print("DocumentIngestionPipeline class defined successfully.")

## 8. Step 6: Synthetic Document Generator for Offline Verification

To test the Phase 1 algorithms deterministically without relying on attached USB webcams or manual file uploads, we implement a realistic synthetic test generator:
* Generates a textured wooden desk background.
* Synthesizes a white document page with realistic non-uniform lighting gradient.
* Renders handwritten-style sample text.
* Warps the document onto the desk using a known homography to simulate a 3D tilted camera capture.

In [ ]:
def generate_synthetic_tilted_document(width=1280, height=720):
    """
    Generates a realistic camera capture of a tilted document on a desk.
    Returns: (synthetic_photo, true_quad_corners)
    """
    # 1. Desk background with texture
    desk = np.full((height, width, 3), (35, 45, 55), dtype=np.uint8)
    noise = np.random.normal(0, 4, (height, width, 3)).astype(np.int16)
    desk = np.clip(desk.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    # 2. Document sheet with lighting gradient
    doc_w, doc_h = 700, 460
    doc = np.full((doc_h, doc_w, 3), (242, 245, 248), dtype=np.uint8)
    grad = np.linspace(0.85, 1.05, doc_w).reshape(1, -1, 1)
    doc = np.clip(doc.astype(np.float32) * grad, 0, 255).astype(np.uint8)
    
    # 3. Handwritten sample annotations
    font = cv2.FONT_HERSHEY_SCRIPT_SIMPLEX
    cv2.putText(doc, "Antigravity Optical System", (40, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (80, 80, 80), 2, cv2.LINE_AA)
    cv2.putText(doc, "Phase 1: Ingestion & Rectification", (40, 130), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (100, 100, 100), 2, cv2.LINE_AA)
    cv2.putText(doc, "The quick brown fox", (45, 220), font, 1.8, (25, 25, 25), 4, cv2.LINE_AA)
    cv2.putText(doc, "jumps over 40928 dogs", (45, 330), font, 1.8, (20, 20, 20), 4, cv2.LINE_AA)
    cv2.putText(doc, "CLASSICAL CV + SHALLOW MLP", (45, 410), cv2.FONT_HERSHEY_SIMPLEX, 0.85, (40, 40, 40), 2, cv2.LINE_AA)
    
    # 4. Warp document into tilted camera perspective
    src_pts = np.array([[0, 0], [doc_w, 0], [doc_w, doc_h], [0, doc_h]], dtype="float32")
    dst_pts = np.array([[220, 110], [1060, 80], [1120, 620], [160, 580]], dtype="float32")
    
    M = cv2.getPerspectiveTransform(src_pts, dst_pts)
    warped_doc = cv2.warpPerspective(doc, M, (width, height), flags=cv2.INTER_LINEAR)
    
    # Composite document over desk
    mask = np.zeros((height, width), dtype=np.uint8)
    cv2.fillConvexPoly(mask, dst_pts.astype(np.int32), 255)
    desk_bg = cv2.bitwise_and(desk, desk, mask=cv2.bitwise_not(mask))
    synthetic_frame = cv2.add(desk_bg, warped_doc)
    
    return synthetic_frame, dst_pts

print("generate_synthetic_tilted_document defined successfully.")

## 9. Interactive Demonstration: Running Phase 1 End-to-End

Here we execute the complete Phase 1 pipeline on a tilted capture and visualize:
1. **The raw input photograph** with detected corner overlay (green quad + blue corner hubs).
2. **The rectified document** ready for Phase 2 illumination-robust preprocessing.

In [ ]:
# 1. Initialize Pipeline
pipeline = DocumentIngestionPipeline(enable_warp=True, min_contour_area_ratio=0.15)

# 2. Acquire test capture
sample_camera_photo, ground_truth_corners = generate_synthetic_tilted_document()

# 3. Run Phase 1
orig_img, rectified_doc, meta = pipeline.process(sample_camera_photo)

# 4. Print telemetry
print("=" * 60)
print("PHASE 1 INGESTION TELEMETRY")
print("=" * 60)
print(f"Raw Capture Dimensions    : {orig_img.shape[1]}x{orig_img.shape[0]} (Channels: {orig_img.shape[2]})")
print(f"Rectified Page Dimensions : {rectified_doc.shape[1]}x{rectified_doc.shape[0]}")
print(f"Warp Applied              : {meta['warp_applied']}")
if meta['corners'] is not None:
    print(f"Detected Corner Polygon   :\n{np.round(meta['corners'], 1)}")
print("=" * 60)

# 5. Visual Display
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Input photograph with quad overlay
annotated_orig = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)
if meta["warp_applied"] and meta["corners"] is not None:
    pts = meta["corners"].astype(np.int32)
    for i in range(4):
        cv2.line(annotated_orig, tuple(pts[i]), tuple(pts[(i + 1) % 4]), (0, 255, 0), 3)
        cv2.circle(annotated_orig, tuple(pts[i]), 8, (255, 0, 0), -1)

axes[0].imshow(annotated_orig)
axes[0].set_title("Input Camera Photo (With Detected Quad Overlays)", fontsize=12, fontweight="bold")
axes[0].axis("off")

# Right: Rectified document surface
axes[1].imshow(cv2.cvtColor(rectified_doc, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"Rectified Document Surface ({rectified_doc.shape[1]}x{rectified_doc.shape[0]})", fontsize=12, fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 10. Verification of the Graceful Bypass Mode

Next, let's verify how Phase 1 handles an image that has **no detectable paper boundaries** (e.g. a tight zoom-in where the text occupies the entire frame with no background desk).

In [ ]:
# Create a tight close-up crop containing text with NO page boundary
close_up = rectified_doc[50:350, 40:600].copy()

# Process through pipeline with auto-warp enabled
orig_close, processed_close, meta_close = pipeline.process(close_up)

# Verify the graceful bypass was triggered
assert not meta_close["warp_applied"], "Warp should be bypassed when no boundary is present!"
assert orig_close.shape == processed_close.shape, "Dimensions should match exactly in bypass mode!"
print("[Verification Passed] Engine gracefully bypassed warp and safely passed the full image forward.")

# Visual display of bypass
plt.figure(figsize=(8, 4))
plt.imshow(cv2.cvtColor(processed_close, cv2.COLOR_BGR2RGB))
plt.title(f"Graceful Bypass Output (Dimensions Preserved: {processed_close.shape[1]}x{processed_close.shape[0]})")
plt.axis("off")
plt.show()

## 11. Summary of Phase 1 Deliverables

| Component | Purpose & Implementation | Status |
| :--- | :--- | :--- |
| **Image Loading** | `cv2.imread` with comprehensive file existence and decode checks. | Verified |
| **Memory Copy** | Preserves untouched original `original_copy = image.copy()` for final annotations. | Verified |
| **Boundary Detection** | Downscaled Canny edge detection + Douglas-Peucker convex 4-point polygon approximation. | Verified |
| **Point Ordering** | Deterministic Top-Left, Top-Right, Bottom-Right, Bottom-Left sorting via coordinate sum & difference. | Verified |
| **Homography Warp** | Euclidean dimension calculation + `cv2.getPerspectiveTransform` + `cv2.warpPerspective`. | Verified |
| **Graceful Bypass** | Automatic fall-through when no 4 corners are found, preventing crashes or distorted warps. | Verified |
| **Handover to Phase 2** | Clean rectified RGB/BGR image output ready for illumination equalization and Sauvola binarization. | Ready |